# SDSTrack on VisEvent — Colab Notebook

This notebook reproduces **SDSTrack** evaluation on the **VisEvent** dataset for the Pattern Recognition course project (Topic #65).

## Workflow
1. **Environment Setup** — Mount Drive, install deps, clone SDSTrack, apply compatibility patches
2. **Dataset Preparation** — Stream VisEvent **test** set from Hugging Face (webdataset tar shards) — download one shard, extract, evaluate, delete
3. **Model Preparation** — Set up pretrained checkpoint, apply PyTorch 2.x compatibility fixes
4. **Streaming Evaluation** — Process test sequences in batches, save results directly to Drive
5. **Metrics** — Compute Success AUC and Precision @ 20px

## Expected Results
- **Success AUC:** ~0.62
- **Precision (20px):** ~0.74

## Tips
- All setup cells are **idempotent** — safe to re-run.
- Evaluation is **resumable** — progress is saved to Drive after every batch.
- Disk usage stays under ~5 GB at any time (one tar shard + extracted sequences).
- Use *Runtime > Factory reset runtime* if you need a completely fresh start.

## 本 Notebook 解决什么问题？

**课题：** 基于事件相机的目标跟踪（Topic #65）

### 核心难点
传统帧相机在高速运动、极端光照变化等场景下存在运动模糊、信息延迟等问题。事件相机以像素级异步触发机制提供微秒级时间分辨率与高动态范围，但如何将其稀疏事件流与传统图像有效融合，是提升跟踪鲁棒性的关键。

### SDSTrack 复现痛点
- **环境断层**：原代码基于 PyTorch 1.11 + Python 3.8，Colab 已升级到 PyTorch 2.x + Python 3.10+
- **路径硬编码**：数据集、模型路径写死为作者本地服务器路径
- **checkpoint 兼容性**：PyTorch 2.6+ 默认 `weights_only=True`，无法加载旧模型
- **数据规模**：VisEvent 测试集 284 GB（webdataset 形式托管在 Hugging Face），Colab 磁盘无法一次性容纳

### 本 Notebook 的解决方案
| 问题 | 解决方案 |
|------|----------|
| 环境不兼容 | 自动检测并应用 PyTorch 2.x / Python 3.10+ 兼容性补丁（`collections.abc`、`torch._six`、`weights_only` 等） |
| 路径错误 | 自动重写 `local.py` 和测试脚本中的硬编码路径为 Colab 路径 |
| 数据集过大 | **流式处理**：逐个下载 tar shard → 提取序列 → 评测 → 删除，磁盘始终 < 5 GB |
| 结果易丢失 | 评测结果实时写入 Google Drive（通过 symlink），支持断点续跑 |
| 模型获取 | OSTrack 预训练模型和 SDSTrack checkpoint 通过 Drive 快捷方式获取（可迁移至 Hugging Face） |
| 长时间运行 | 内置 keepalive 线程 + 每 batch 自动保存进度到 Drive |
| 指标计算 | 自动读取 tracking results，计算 **Success AUC** 和 **Precision @ 20px** |

### 最终目标
在 VisEvent 测试集上复现 SDSTrack，获得量化的跟踪性能指标，为课程设计报告提供实验数据支撑。

In [ ]:
# ============================================================
# P0: Configuration & State Check
# ============================================================

import os

CONFIG = {
    "MOUNT_POINT": "/content/drive",
    "WS": "/content/sdstrack",
    "BASE_DRIVE": "/content/drive/MyDrive/EvTrack",
    "DATA_DIR": "/content/sdstrack/data",
    "VISEVENT_DIR": "/content/sdstrack/data/visevent",
    "HF_DATASET": "krisspy39/visevent",
    "PRETRAINED_DIR": "/content/sdstrack/pretrained/vitb_256_mae_ce_32x4_ep300",
    "MODELS_DIR": "/content/sdstrack/models",
    "CHECKPOINT_DIR": "/content/sdstrack/output/checkpoints/train/sdstrack/cvpr2024_rgbe",
    "PRETRAINED_MODEL": "/content/sdstrack/pretrained/vitb_256_mae_ce_32x4_ep300/OSTrack_ep0300.pth.tar",
    "EVAL_MODEL": "/content/sdstrack/models/SDSTrack_cvpr2024_rgbe.pth.tar",
    "TEST_DIR": "/content/sdstrack/data/visevent/test/test_subset",
    "RESULTS_DIR": "/content/sdstrack/RGBE_workspace/results/VisEvent/cvpr2024_rgbe",
    "DRIVE_RESULTS": "/content/drive/MyDrive/EvTrack/sdstrack_results/VisEvent/cvpr2024_rgbe",
    "DRIVE_PROGRESS": "/content/drive/MyDrive/EvTrack/sdstrack_progress.json",
    "CLONE_FLAG": "/content/sdstrack/.upstream_cloned",
    "PATCH_FLAG": "/content/sdstrack/.patches_applied",
    "PATH_FLAG": "/content/sdstrack/.paths_fixed",
}

def check(path):
    return os.path.exists(path)

def count_subdirs(path):
    if not check(path):
        return 0
    try:
        return len([d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d))])
    except Exception:
        return 0

result_count = 0
if check(CONFIG["RESULTS_DIR"]):
    result_count = len([f for f in os.listdir(CONFIG["RESULTS_DIR"]) if f.endswith(".txt")])

checks = {
    "Google Drive mounted": check(os.path.join(CONFIG["MOUNT_POINT"], "MyDrive")),
    "SDSTrack cloned": check(os.path.join(CONFIG["WS"], "lib")),
    "Patches applied": check(CONFIG["PATCH_FLAG"]),
    "Paths fixed": check(CONFIG["PATH_FLAG"]),
    "Pretrained model": check(CONFIG["PRETRAINED_MODEL"]),
    "Eval model ready": check(CONFIG["EVAL_MODEL"]),
    "Results on Drive": result_count > 0,
}

print("=" * 50)
print("SDSTrack Setup State")
print("=" * 50)
for name, ok in checks.items():
    print(f"  {'OK' if ok else '--'} {name}")
print(f"  Result files: {result_count}")
done = sum(checks.values())
print("=" * 50)
print(f"Progress: {done}/{len(checks)}")
if done == len(checks):
    print("All set! Proceed to Phase 5: Metrics.")
elif not checks["Google Drive mounted"]:
    print("Next: Phase 1 -> Mount Google Drive")
elif not checks["SDSTrack cloned"]:
    print("Next: Phase 1 -> Environment Setup")
elif not checks["Eval model ready"]:
    print("Next: Phase 3 -> Model Preparation")
else:
    print("Next: Phase 4 -> Streaming Evaluation")

## Phase 1: Environment Setup

Set up the SDSTrack codebase and its dependencies.

**Notes:**
- Colab provides PyTorch 2.x (upstream requires 1.11.0, but we apply compatibility patches).
- All cells are idempotent — safe to re-run.

In [ ]:
# ============================================================
# P1.1 Mount Google Drive
# ============================================================
import os
from google.colab import drive
MOUNT_POINT = "/content/drive"
if os.path.ismount(MOUNT_POINT) and os.path.exists(f"{MOUNT_POINT}/MyDrive"):
    print("Google Drive already mounted.")
else:
    print("Mounting Google Drive...")
    drive.mount(MOUNT_POINT, force_remount=False)
    print("Drive mounted successfully!")

In [ ]:
# ============================================================
# P1.2 GPU and CUDA Verification
# ============================================================
import torch
print("=" * 50)
print("GPU / CUDA Info")
print("=" * 50)
!nvidia-smi -L
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
print("\nPyTorch CUDA Info")
print("-" * 50)
print(f"PyTorch version:  {torch.__version__}")
print(f"CUDA available:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA (PyTorch):   {torch.version.cuda}")
    print(f"GPU:              {torch.cuda.get_device_name(0)}")
    print(f"GPU memory:       {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    x = torch.rand(500, 500).cuda()
    y = torch.mm(x, x.t())
    print(f"\nGPU tensor test:  OK (shape {y.shape})")
else:
    print("\nWARNING: CUDA not available!")

In [ ]:
# ============================================================
# P1.3 Install SDSTrack Dependencies
# ============================================================
import subprocess, sys, importlib
print("Checking dependencies...")
PKGS = [
    ('PyYAML', 'yaml'), ('easydict', 'easydict'), ('cython', 'cython'),
    ('opencv-python', 'cv2'), ('pandas', 'pandas'), ('pycocotools', 'pycocotools'),
    ('jpeg4py', 'jpeg4py'), ('scipy', 'scipy'), ('timm==0.5.4', 'timm'),
    ('tb-nightly', 'tensorboard'), ('lmdb', 'lmdb'), ('visdom', 'visdom'),
    ('wandb', 'wandb'), ('vot-toolkit==0.5.3', 'vot_toolkit'),
    ('vot-trax==3.0.3', 'vot_trax'), ('tqdm', 'tqdm'),
    ('huggingface-hub', 'huggingface_hub'),
]
missing = []
for pkg, mod in PKGS:
    try:
        importlib.import_module(mod.replace('-', '_'))
        print(f"  OK {pkg.split('==')[0]}")
    except ImportError:
        print(f"  -- {pkg.split('==')[0]}")
        missing.append(pkg)
if missing:
    print(f"\nInstalling {len(missing)} missing packages...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    print("Installation complete.")
else:
    print("\nAll dependencies already installed.")

In [ ]:
# ============================================================
# P1.4 Verify Installation
# ============================================================
import torch, cv2, yaml, timm, scipy
print("=" * 50)
print("Dependency Versions")
print("=" * 50)
print(f"PyTorch:  {torch.__version__}")
print(f"CUDA:     {torch.cuda.is_available()} ({torch.version.cuda if torch.cuda.is_available() else 'N/A'})")
print(f"OpenCV:   {cv2.__version__}")
print(f"timm:     {timm.__version__}")
print(f"scipy:    {scipy.__version__}")
if torch.cuda.is_available():
    x = torch.rand(1000, 1000).cuda()
    y = torch.mm(x, x.t())
    print(f"\nGPU test: OK (result {y.shape})")
else:
    print("\nNo GPU detected!")

In [ ]:
# ============================================================
# P1.5 Clone SDSTrack & Apply All Patches
# Idempotent: skips clone/patch if already done.
# ============================================================
import os, shutil
WS = "/content/sdstrack"
os.makedirs(WS, exist_ok=True)
os.chdir(WS)
UPSTREAM = os.path.join(WS, ".upstream_cloned")
PATCHED = os.path.join(WS, ".patches_applied")
PATHS = os.path.join(WS, ".paths_fixed")

# --- Clone ---
if os.path.exists(UPSTREAM) and os.path.exists(os.path.join(WS, "lib")):
    print("SDSTrack already cloned.")
else:
    print("Cloning SDSTrack...")
    !git clone https://github.com/hoqolo/SDSTrack.git .
    with open(UPSTREAM, "w") as f: f.write("done")
    print("Clone complete.")

# --- Apply PyTorch 2.x / Python 3.10+ patches ---
if os.path.exists(PATCHED):
    print("Patches already applied.")
else:
    loader = os.path.join(WS, "lib", "train", "data", "loader.py")
    if os.path.exists(loader):
        print("Applying compatibility patches...")
        with open(loader, "r") as f:
            c = f.read()
        if "import collections.abc" not in c:
            c = c.replace("import collections", "import collections\nimport collections.abc")
        c = c.replace("from torch._six import string_classes",
                      "try:\n    from torch._six import string_classes\nexcept ImportError:\n    string_classes = (str, bytes)")
        c = c.replace("collections.Mapping", "collections.abc.Mapping")
        c = c.replace("collections.Sequence", "collections.abc.Sequence")
        with open(loader, "w") as f: f.write(c)

        patched_files = []
        for root, dirs, files in os.walk(os.path.join(WS, "lib")):
            for fname in files:
                if not fname.endswith(".py"): continue
                fpath = os.path.join(root, fname)
                with open(fpath, "r") as f:
                    content = f.read()
                original = content
                content = content.replace("map_location='cpu')", "map_location='cpu', weights_only=False)")
                content = content.replace('map_location="cpu")', 'map_location="cpu", weights_only=False)')
                if content != original:
                    with open(fpath, "w") as f: f.write(content)
                    patched_files.append(fpath)
        with open(PATCHED, "w") as f: f.write("done")
        print(f"Patched loader.py")
        print(f"Patched weights_only in {len(patched_files)} files")
    else:
        print("loader.py not found. Clone may have failed.")

# --- Fix paths ---
if os.path.exists(PATHS):
    print("Paths already fixed.")
else:
    print("Configuring paths...")
    !python tracking/create_default_local_file.py --workspace_dir . --data_dir ./data --save_dir ./output

    # Fix local.py (train)
    local_py = os.path.join(WS, "lib", "train", "admin", "local.py")
    if os.path.exists(local_py):
        with open(local_py, "r") as f: c = f.read()
        c = c.replace("self.visevent_dir = '/home/houxiaojun/Workspace/SDSTrack/data/visevent/train'",
                      "self.visevent_dir = '/content/sdstrack/data/visevent/train/train_subset'")
        with open(local_py, "w") as f: f.write(c)
        print("Fixed visevent_dir (train)")

    # Fix local.py (test/eval)
    local_py_test = os.path.join(WS, "lib", "test", "evaluation", "local.py")
    if os.path.exists(local_py_test):
        with open(local_py_test, "r") as f: c = f.read()
        c = c.replace("/home/houxiaojun/Workspace/SDSTrack", WS)
        with open(local_py_test, "w") as f: f.write(c)
        print("Fixed local.py (test)")

    # Fix config.py
    config_py = os.path.join(WS, "lib", "config", "sdstrack", "config.py")
    if os.path.exists(config_py):
        with open(config_py, "r") as f: c = f.read()
        c = c.replace("/home/houxiaojun/Workspace/SDSTrack/experiments/", "/content/sdstrack/experiments/")
        with open(config_py, "w") as f: f.write(c)
        print("Fixed config.py")

    # Fix parameter file
    param_py = os.path.join(WS, "lib", "test", "parameter", "sdstrack.py")
    if os.path.exists(param_py):
        with open(param_py, "r") as f: c = f.read()
        c = c.replace("/home/houxiaojun/Workspace/SDSTrack/experiments/", "/content/sdstrack/experiments/")
        with open(param_py, "w") as f: f.write(c)
        print("Fixed parameter/sdstrack.py")

    # Fix test script path
    test_script = os.path.join(WS, "RGBE_workspace", "test_rgbe_mgpus.py")
    if os.path.exists(test_script):
        with open(test_script, "r") as f: c = f.read()
        c = c.replace("seq_home = '/public/datasets_neo/VisEvent/VisEvent_dataset/testset/test_subset'",
                      "seq_home = '/content/sdstrack/data/visevent/test/test_subset'")
        with open(test_script, "w") as f: f.write(c)
        print("Fixed test script path")

    with open(PATHS, "w") as f: f.write("done")
    print("Paths configured.")

In [ ]:
# ============================================================
# P1.6 Test Imports
# ============================================================
import sys
sys.path.insert(0, "/content/sdstrack")
tests = [
    ("lib.train.data.loader", "LTRLoader"),
    ("lib.test.evaluation", "create_default_local_file_test"),
    ("lib.train.admin.local", "EnvironmentSettings"),
]
all_ok = True
for mod, obj in tests:
    try:
        m = __import__(mod, fromlist=[obj])
        getattr(m, obj)
        print(f"  OK {mod}.{obj}")
    except Exception as e:
        print(f"  FAIL {mod}.{obj}: {e}")
        all_ok = False
if all_ok:
    from lib.train.admin import local
    print(f"\nVisEvent train dir: {local.EnvironmentSettings().visevent_dir}")
    print("\nEnvironment ready!")
else:
    print("\nSome imports failed. Check earlier cells.")

In [ ]:
# ============================================================
# P1.7 Prepare Data Directory & Results Symlink to Drive
# Idempotent: handles existing directories/links safely.
# Results are written directly to Drive so they survive Colab disconnects.
# ============================================================
import os, shutil
WS = "/content/sdstrack"
DATA_DIR = os.path.join(WS, "data")
os.makedirs(DATA_DIR, exist_ok=True)

# Data dir (visevent)
DRIVE_DATA = "/content/drive/MyDrive/EvTrack/datasets/VisEvent"
link = os.path.join(DATA_DIR, "visevent")
if os.path.islink(link) and os.path.exists(link):
    print(f"Data symlink OK -> {os.readlink(link)}")
elif os.path.isdir(link) and not os.path.islink(link):
    shutil.rmtree(link)
    if os.path.exists(DRIVE_DATA):
        os.symlink(DRIVE_DATA, link)
    else:
        os.makedirs(link, exist_ok=True)
        print(f"Created empty data dir: {link}")
elif not os.path.exists(link):
    if os.path.exists(DRIVE_DATA):
        os.symlink(DRIVE_DATA, link)
        print(f"Data symlink -> {DRIVE_DATA}")
    else:
        os.makedirs(link, exist_ok=True)
        print(f"Created empty data dir: {link}")

# Results symlink to Drive
DRIVE_RESULTS = "/content/drive/MyDrive/EvTrack/sdstrack_results"
LOCAL_RESULTS = os.path.join(WS, "RGBE_workspace", "results")
os.makedirs(DRIVE_RESULTS, exist_ok=True)
os.makedirs(os.path.dirname(LOCAL_RESULTS), exist_ok=True)
if os.path.exists(LOCAL_RESULTS) and not os.path.islink(LOCAL_RESULTS):
    shutil.rmtree(LOCAL_RESULTS)
if not os.path.exists(LOCAL_RESULTS):
    os.symlink(DRIVE_RESULTS, LOCAL_RESULTS)
    print(f"Results symlink -> {DRIVE_RESULTS}")
else:
    print(f"Results symlink OK -> {os.readlink(LOCAL_RESULTS)}")

print("\nData directory:")
for item in sorted(os.listdir(DATA_DIR)):
    p = os.path.join(DATA_DIR, item)
    kind = "[LINK]" if os.path.islink(p) else "[DIR]" if os.path.isdir(p) else "[FILE]"
    print(f"  {item:20s} {kind}")

In [ ]:
# ============================================================
# P1.8 Download OSTrack Pretrained Model
# Idempotent: skips if model already exists.
# NOTE: Add 'OSTrack_ep0300.pth.tar' as a shortcut to MyDrive.
# ============================================================
import os, shutil
PRETRAINED_DIR = "/content/sdstrack/pretrained/vitb_256_mae_ce_32x4_ep300"
expected = os.path.join(PRETRAINED_DIR, "OSTrack_ep0300.pth.tar")
shortcut = "/content/drive/MyDrive/OSTrack_ep0300.pth.tar"
if os.path.exists(expected):
    mb = os.path.getsize(expected) / (1024 * 1024)
    print(f"Model already exists ({mb:.1f} MB)")
elif os.path.exists(shortcut):
    os.makedirs(PRETRAINED_DIR, exist_ok=True)
    shutil.copy2(shortcut, expected)
    mb = os.path.getsize(expected) / (1024 * 1024)
    print(f"Copied from Drive shortcut ({mb:.1f} MB)")
else:
    print("Model not found.")
    print("\nPlease add a shortcut to your Google Drive:")
    print("  1. Open: https://drive.google.com/drive/folders/1ttafo0O5S9DXK2PX0YqPvPrQ-HWJjhSy?usp=sharing")
    print("  2. Right-click 'OSTrack_ep0300.pth.tar'")
    print("  3. Select 'Organize' -> 'Add shortcut' -> 'All locations' -> 'My Drive' -> 'Add'")
    print("  4. Re-run this cell")

## Phase 2: Dataset Preparation

The VisEvent test set is **284 GB** as webdataset tar shards on Hugging Face. Colab disk (~78 GB) cannot hold it all.

**Solution:** We process **one tar shard at a time**:
1. Download a single tar shard (~1–2 GB)
2. Extract its sequences to disk
3. Run evaluation on those sequences
4. Delete extracted images to free space
5. Repeat for the next shard

**Disk usage stays under ~5 GB at all times.**

| Set | Source | Size | Needed? |
|-----|--------|------|---------|
| Train | Hugging Face | ~383 GB | **No** (training cancelled, Issue #4) |
| Test  | Hugging Face | ~284 GB | **Yes** (streamed shard-by-shard) |


In [ ]:
# ============================================================
# P2.1 List Test Tar Files from Hugging Face
# Fetches the list of tar shard filenames via HF API.
# Idempotent: uses local cache if already fetched.
# ============================================================
import os, json, requests
from huggingface_hub import hf_hub_download

WS = "/content/sdstrack"
DATA_DIR = os.path.join(WS, "data", "visevent")
TEST_TAR_DIR = os.path.join(DATA_DIR, "hf_test_tars")
os.makedirs(TEST_TAR_DIR, exist_ok=True)

HF_REPO = "krisspy39/visevent"
TAR_LIST_CACHE = os.path.join(TEST_TAR_DIR, "test_tar_list.json")

if os.path.exists(TAR_LIST_CACHE):
    with open(TAR_LIST_CACHE) as f:
        tar_files = json.load(f)
    print(f"Loaded {len(tar_files)} tar files from cache")
else:
    print("Fetching tar file list from Hugging Face API...")
    api_url = f"https://huggingface.co/api/datasets/{HF_REPO}/tree/main/webdataset/test"
    resp = requests.get(api_url)
    resp.raise_for_status()
    data = resp.json()
    tar_files = [f["path"].replace("webdataset/test/", "") for f in data
                 if f["type"] == "file" and f["path"].endswith(".tar")]
    tar_files.sort()
    with open(TAR_LIST_CACHE, "w") as f:
        json.dump(tar_files, f)
    print(f"Fetched {len(tar_files)} tar files")

total_size = 0
for t in tar_files[:5]:
    print(f"  {t}")
print(f"  ... ({len(tar_files)} total)")

In [ ]:
# ============================================================
# P2.2 Setup Results Directory on Google Drive
# Ensures the results symlink (created in P1.7) points to a valid Drive path.
# ============================================================
import os
DRIVE_RESULTS = "/content/drive/MyDrive/EvTrack/sdstrack_results/VisEvent/cvpr2024_rgbe"
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f"Results directory on Drive: {DRIVE_RESULTS}")
files = [f for f in os.listdir(DRIVE_RESULTS) if f.endswith(".txt")]
print(f"Existing result files: {len(files)}")

## Phase 3: Model Preparation

Prepare the pretrained checkpoint for evaluation and apply PyTorch 2.x compatibility patches.

**Which model?** We are evaluating on **VisEvent** (RGB-E / event camera), so we need the **`cvpr2024_rgbe`** checkpoint.

| Config | Modality | Dataset |
|--------|----------|---------|
| `cvpr2024_rgbd` | RGB-D | DepthTrack |
| **`cvpr2024_rgbe`** | **RGB-E** | **VisEvent** ← **ours** |
| `cvpr2024_rgbt` | RGB-T | LaSHeR |

In [ ]:
# ============================================================
# P3.1 Prepare Model Checkpoint
# Idempotent: skips if already set up.
# Downloads from Hugging Face Hub. Falls back to Drive shortcuts.
# ============================================================
import os, shutil
from huggingface_hub import hf_hub_download

WS = "/content/sdstrack"
os.chdir(WS)
MODEL_DIR = os.path.join(WS, "models")
CKPT_DIR = os.path.join(WS, "output", "checkpoints", "train", "sdstrack", "cvpr2024_rgbe")
SYMLINK = os.path.join(MODEL_DIR, "SDSTrack_cvpr2024_rgbe.pth.tar")
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

if os.path.exists(SYMLINK):
    real = os.path.realpath(SYMLINK)
    mb = os.path.getsize(real) / (1024 * 1024)
    print(f"Model ready ({mb:.1f} MB)")
    print(f"   {SYMLINK}")
    raise SystemExit

# Try Hugging Face first
HF_MODEL_REPO = "krisspy39/sdstrack-rgbe"
HF_FILENAME = "SDSTrack_cvpr2024_rgbe.pth.tar"

try:
    print(f"Downloading from Hugging Face: {HF_MODEL_REPO}...")
    downloaded = hf_hub_download(
        repo_id=HF_MODEL_REPO,
        filename=HF_FILENAME,
        repo_type="model",
        local_dir=CKPT_DIR,
    )
    # The file is downloaded as a blob; create the expected symlink
    dest = os.path.join(CKPT_DIR, HF_FILENAME)
    # hf_hub_download with local_dir may create symlinks; find the actual file
    if os.path.islink(dest):
        real_file = os.readlink(dest)
    else:
        real_file = dest
    os.symlink(real_file, SYMLINK)
    mb = os.path.getsize(real_file) / (1024 * 1024)
    print(f"Downloaded from HF ({mb:.1f} MB)")
    print(f"   {SYMLINK} -> {real_file}")
except Exception as e:
    print(f"HF download failed: {e}")
    print("Falling back to Google Drive shortcuts...")
    
    # Fallback: Search Drive shortcuts
    NAMES = ["SDSTrack_cvpr2024_rgbe.pth.tar", "SDSTrack_ep0050.pth.tar"]
    found = None
    for name in NAMES:
        for path in [f"/content/drive/MyDrive/{name}", f"/content/drive/MyDrive/SDSTrack_models/{name}"]:
            if os.path.exists(path):
                found = path
                break
        if found: break
    
    if found:
        dest = os.path.join(CKPT_DIR, "SDSTrack_ep0050.pth.tar")
        shutil.copy2(found, dest)
        os.symlink(dest, SYMLINK)
        mb = os.path.getsize(dest) / (1024 * 1024)
        print(f"Copied from Drive ({mb:.1f} MB)")
        print(f"   {SYMLINK} -> {dest}")
    else:
        print("Model checkpoint not found.")
        print("\nPlease either:")
        print("  1. Log in to Hugging Face: !huggingface-cli login")
        print("  2. Or add a Drive shortcut: Right-click 'SDSTrack_cvpr2024_rgbe.pth.tar' -> Add shortcut -> My Drive")
        print("  3. Re-run this cell")

In [ ]:
# ============================================================
# P3.2 Extra PyTorch 2.x Compatibility Check
# ============================================================
import os
WS = "/content/sdstrack"
SPECIFIC_PATCHES = [
    ("lib/test/tracker/sdstrack.py",
     'checkpoint = torch.load(self.params.checkpoint + \'.tar\', map_location="cpu")',
     'checkpoint = torch.load(self.params.checkpoint + \'.tar\', map_location="cpu", weights_only=False)'),
    ("lib/train/trainers/base_trainer.py",
     "checkpoint_dict = torch.load(checkpoint_path, map_location='cpu')",
     "checkpoint_dict = torch.load(checkpoint_path, map_location='cpu', weights_only=False)"),
]
for filepath, old, new in SPECIFIC_PATCHES:
    full = os.path.join(WS, filepath)
    if not os.path.exists(full):
        print(f"File not found: {filepath}")
        continue
    with open(full, "r") as f:
        content = f.read()
    if old in content:
        content = content.replace(old, new)
        with open(full, "w") as f:
            f.write(content)
        print(f"Patched {filepath}")
    else:
        print(f"Already patched or not found: {filepath}")
print("\nDone. Proceed to Phase 4: Streaming Evaluation.")

## Phase 4: Streaming Evaluation

Download one tar shard at a time from Hugging Face, extract sequences, evaluate them, and delete to free disk space.

**Resumability:**
- Progress is saved to `MyDrive/EvTrack/sdstrack_progress.json` after every shard
- Result files are saved directly to Drive via symlink
- If Colab disconnects, re-run this cell and it will skip completed shards

**Expected time:** 4–8 hours total (depends on GPU and connection).

**Disk usage:** ~3–5 GB at any time.

In [ ]:
# ============================================================
# P4.1 Streaming Evaluation Loop
# Processes test tar shards one at a time.
# Keeps disk usage low by deleting extracted sequences after evaluation.
# Saves progress and results to Google Drive for resumability.
# ============================================================

import os
import json
import tarfile
import shutil
import subprocess
import threading
import time
from huggingface_hub import hf_hub_download

WS = "/content/sdstrack"
os.chdir(WS)

DATA_DIR = os.path.join(WS, "data", "visevent")
TEST_OUT = os.path.join(DATA_DIR, "test", "test_subset")
TEST_TAR_DIR = os.path.join(DATA_DIR, "hf_test_tars")
RESULTS_DIR = os.path.join(WS, "RGBE_workspace", "results", "VisEvent", "cvpr2024_rgbe")
DRIVE_PROGRESS = "/content/drive/MyDrive/EvTrack/sdstrack_progress.json"
CACHE_DIR = "/tmp/hf_cache"

os.makedirs(TEST_OUT, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DRIVE_PROGRESS), exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

# Load progress
if os.path.exists(DRIVE_PROGRESS):
    with open(DRIVE_PROGRESS) as f:
        progress = json.load(f)
else:
    progress = {"completed_tars": [], "completed_seqs": []}

# Load tar list
with open(os.path.join(TEST_TAR_DIR, "test_tar_list.json")) as f:
    tar_files = json.load(f)

def get_seq_name(key):
    parts = key.split("__")
    if len(parts) >= 3 and parts[0] == "test_subset":
        return parts[1]
    return None

def seq_is_complete(seq_dir):
    gt = os.path.join(seq_dir, "groundtruth.txt")
    absent = os.path.join(seq_dir, "absent_label.txt")
    vis = os.path.join(seq_dir, "vis_imgs")
    evt = os.path.join(seq_dir, "event_imgs")
    if not all(os.path.exists(p) for p in [gt, absent, vis, evt]):
        return False
    try:
        with open(gt) as f:
            gt_lines = len(f.readlines())
        vis_frames = len([f for f in os.listdir(vis) if f.endswith('.bmp')])
        evt_frames = len([f for f in os.listdir(evt) if f.endswith('.bmp')])
        return vis_frames == gt_lines and evt_frames == gt_lines
    except Exception:
        return False

def result_exists(seq):
    return os.path.exists(os.path.join(RESULTS_DIR, f"{seq}.txt"))

def check_disk():
    total, used, free = shutil.disk_usage("/")
    print(f"  [Disk] Free: {free / 1e9:.1f} GB")
    return free > 5e9

# Detect GPU threads
try:
    import torch
    gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else ""
    threads = 4 if any(g in gpu_name for g in ["A100", "V100", "L4"]) else 1
except Exception:
    threads = 1
print(f"Using threads={threads} for evaluation")

BATCH_SIZE = 10  # evaluate when we have this many new sequences
pending_seqs = set()
total_completed = len(progress["completed_seqs"])

# Keepalive thread
keepalive_running = True
def keepalive():
    start = time.time()
    while keepalive_running:
        time.sleep(60)
        elapsed = time.time() - start
        current = 0
        if os.path.exists(RESULTS_DIR):
            current = len([f for f in os.listdir(RESULTS_DIR) if f.endswith(".txt")])
        print(f"\n[{elapsed/60:.0f} min keepalive] Results: {current} sequences done\n", flush=True)
t = threading.Thread(target=keepalive, daemon=True)
t.start()

try:
    for i, tar_name in enumerate(tar_files):
        if tar_name in progress["completed_tars"]:
            continue

        print("\n" + "=" * 60)
        print(f"[{i+1}/{len(tar_files)}] Processing {tar_name}")
        print("=" * 60)

        if not check_disk():
            print("WARNING: Low disk space! Deleting extracted sequences...")
            for s in os.listdir(TEST_OUT):
                p = os.path.join(TEST_OUT, s)
                if os.path.isdir(p) and result_exists(s):
                    shutil.rmtree(p, ignore_errors=True)

        # Download tar to custom cache (cleared after each tar)
        print("  Downloading...")
        try:
            tar_path = hf_hub_download(
                repo_id="krisspy39/visevent",
                filename=f"webdataset/test/{tar_name}",
                repo_type="dataset",
                cache_dir=CACHE_DIR,
            )
        except Exception as e:
            print(f"  ERROR downloading {tar_name}: {e}")
            print("  Will retry on next run.")
            continue

        print(f"  Downloaded -> {tar_path}")

        # List and extract sequences from tar
        print("  Extracting sequences...")
        with tarfile.open(tar_path, "r") as tf:
            members = tf.getmembers()
            tar_seqs = {}
            for m in members:
                if not m.isfile():
                    continue
                seq = get_seq_name(m.name)
                if seq:
                    if seq not in tar_seqs:
                        tar_seqs[seq] = []
                    tar_seqs[seq].append(m)

            new_seqs = []
            for seq, seq_members in tar_seqs.items():
                if result_exists(seq):
                    continue
                seq_dir = os.path.join(TEST_OUT, seq)
                os.makedirs(seq_dir, exist_ok=True)
                for m in seq_members:
                    parts = m.name.split("__")
                    if len(parts) < 3:
                        continue
                    rest = "__".join(parts[2:]).replace("__", "/")
                    out_path = os.path.join(seq_dir, rest)
                    if not os.path.exists(out_path):
                        os.makedirs(os.path.dirname(out_path), exist_ok=True)
                        fobj = tf.extractfile(m)
                        if fobj:
                            with open(out_path, "wb") as out_f:
                                out_f.write(fobj.read())
                if seq_is_complete(seq_dir):
                    pending_seqs.add(seq)
                    new_seqs.append(seq)
                    print(f"    Extracted complete: {seq}")

        # Clear cache to free space
        shutil.rmtree(CACHE_DIR)
        os.makedirs(CACHE_DIR, exist_ok=True)
        print(f"  Cache cleared. Pending sequences: {len(pending_seqs)}")

        # Evaluate if we have enough pending sequences
        if len(pending_seqs) >= BATCH_SIZE or (i == len(tar_files) - 1 and pending_seqs):
            batch = sorted(pending_seqs)
            testlist_path = os.path.join(TEST_OUT, "testlist.txt")
            with open(testlist_path, "w") as f:
                for s in batch:
                    f.write(s + "\n")

            print(f"\n  Evaluating {len(batch)} sequences...")
            proc = subprocess.Popen(
                ["python", "./RGBE_workspace/test_rgbe_mgpus.py",
                 "--script_name", "sdstrack",
                 "--num_gpus", "1",
                 "--threads", str(threads),
                 "--epoch", "50",
                 "--yaml_name", "cvpr2024_rgbe"],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1
            )
            try:
                for line in proc.stdout:
                    print(line, end="", flush=True)
            except KeyboardInterrupt:
                print("\n  Interrupted by user")
            proc.wait()

            done = set()
            for s in batch:
                if result_exists(s):
                    done.add(s)
                    shutil.rmtree(os.path.join(TEST_OUT, s), ignore_errors=True)

            progress["completed_seqs"].extend(sorted(done))
            pending_seqs -= done
            total_completed += len(done)
            print(f"\n  Batch done: {len(done)}/{len(batch)} evaluated. Total: {total_completed}")

        # Mark tar as processed
        progress["completed_tars"].append(tar_name)
        with open(DRIVE_PROGRESS, "w") as f:
            json.dump(progress, f)
        print(f"  Progress saved. Tars done: {len(progress['completed_tars'])}/{len(tar_files)}")

finally:
    keepalive_running = False

# Final evaluation for any remaining pending sequences
if pending_seqs:
    batch = sorted(pending_seqs)
    testlist_path = os.path.join(TEST_OUT, "testlist.txt")
    with open(testlist_path, "w") as f:
        for s in batch:
            f.write(s + "\n")
    print(f"\nFinal evaluation: {len(batch)} remaining sequences...")
    proc = subprocess.Popen(
        ["python", "./RGBE_workspace/test_rgbe_mgpus.py",
         "--script_name", "sdstrack",
         "--num_gpus", "1",
         "--threads", str(threads),
         "--epoch", "50",
         "--yaml_name", "cvpr2024_rgbe"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    for s in batch:
        if result_exists(s):
            progress["completed_seqs"].append(s)
            shutil.rmtree(os.path.join(TEST_OUT, s), ignore_errors=True)
    with open(DRIVE_PROGRESS, "w") as f:
        json.dump(progress, f)

print("\n" + "=" * 60)
print(f"Streaming evaluation complete!")
print(f"Total sequences evaluated: {len(set(progress['completed_seqs']))}")
print(f"Results saved to: {RESULTS_DIR}")
print("=" * 60)

## Phase 5: Metrics & Results

After the streaming evaluation finishes (or any time you want to check progress), compute the final metrics.

In [ ]:
# ============================================================
# P5.1 Check Progress
# ============================================================
import os
RESULTS_DIR = "/content/sdstrack/RGBE_workspace/results/VisEvent/cvpr2024_rgbe"
if os.path.exists(RESULTS_DIR):
    files = [f for f in os.listdir(RESULTS_DIR) if f.endswith(".txt")]
    print(f"Result files: {len(files)}")
    if files:
        print(f"Latest 5: {sorted(files)[-5:]}")
else:
    print("Results directory not found.")

DRIVE_PROGRESS = "/content/drive/MyDrive/EvTrack/sdstrack_progress.json"
if os.path.exists(DRIVE_PROGRESS):
    import json
    with open(DRIVE_PROGRESS) as f:
        p = json.load(f)
    print(f"Tars completed: {len(p['completed_tars'])}")
    print(f"Sequences completed: {len(set(p['completed_seqs']))}")
else:
    print("No progress file found.")

In [ ]:
# ============================================================
# P5.2 Compute Evaluation Metrics
# ============================================================
import os, numpy as np, glob
RESULTS_DIR = "/content/sdstrack/RGBE_workspace/results/VisEvent/cvpr2024_rgbe"
GT_BASE = "/content/sdstrack/data/visevent/test/test_subset"

def compute_iou(box1, box2):
    x1, y1, w1, h1 = box1
    x2, y2, w2, h2 = box2
    xi1 = max(x1, x2); yi1 = max(y1, y2)
    xi2 = min(x1 + w1, x2 + w2); yi2 = min(y1 + h1, y2 + h2)
    inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
    union = w1 * h1 + w2 * h2 - inter
    return inter / union if union > 0 else 0

def compute_metrics(results_dir, gt_base):
    if not os.path.exists(results_dir):
        print("ERROR: Results directory not found."); return
    files = sorted(glob.glob(os.path.join(results_dir, "*.txt")))
    if not files:
        print("ERROR: No result files found."); return
    print(f"Processing {len(files)} result files...")
    all_ious, all_dists = [], []
    for res_file in files:
        seq = os.path.basename(res_file).replace(".txt", "")
        gt_file = os.path.join(gt_base, seq, "groundtruth.txt")
        if not os.path.exists(gt_file):
            print(f"  Skipping {seq} (no groundtruth)"); continue
        try:
            pred = np.loadtxt(res_file, delimiter=",")
            gt = np.loadtxt(gt_file, delimiter=",")
        except Exception:
            print(f"  Skipping {seq} (load error)"); continue
        if pred.ndim == 1: pred = pred.reshape(1, -1)
        if gt.ndim == 1: gt = gt.reshape(1, -1)
        n = min(len(pred), len(gt))
        pred, gt = pred[:n], gt[:n]
        for p, g in zip(pred, gt):
            all_ious.append(compute_iou(p, g))
            pcx, pcy = p[0] + p[2]/2, p[1] + p[3]/2
            gcx, gcy = g[0] + g[2]/2, g[1] + g[3]/2
            all_dists.append(np.sqrt((pcx - gcx)**2 + (pcy - gcy)**2))
    if not all_ious:
        print("No valid data to compute metrics."); return
    all_ious = np.array(all_ious)
    all_dists = np.array(all_dists)
    thresholds = np.arange(0, 1.05, 0.05)
    success_rates = [np.mean(all_ious >= t) for t in thresholds]
    auc = np.mean(success_rates)
    prec_20 = np.mean(all_dists <= 20)
    print("\n" + "=" * 50)
    print("EVALUATION RESULTS")
    print("=" * 50)
    print(f"Success AUC:      {auc:.4f}")
    print(f"Precision (20px): {prec_20:.4f}")
    print(f"Total frames:     {len(all_ious)}")
    print("=" * 50)
    return {"auc": auc, "prec_20": prec_20}

metrics = compute_metrics(RESULTS_DIR, GT_BASE)

## Appendix: Troubleshooting & Notes

**1. `torch.load` weights_only error**
- Already fixed in Phase 1.5 and 3.2. Re-run those cells if you see this error.

**2. Missing `testlist.txt`**
- Phase 4.1 writes it automatically before each evaluation batch.

**3. Evaluation crashes with multiprocessing**
- Phase 4.1 auto-detects GPU and uses `threads=1` for T4. You can also edit the cell to force `threads=1`.

**4. Colab disconnects during evaluation**
- Keep the browser tab active.
- Progress is saved to Drive after every tar shard.
- Re-run Phase 4.1 to resume from the last completed shard.

**5. Model not found**
- **SDSTrack checkpoint** is automatically downloaded from Hugging Face: `krisspy39/sdstrack-rgbe` (see Phase 3.1).
- **OSTrack pretrained model** still requires a Google Drive shortcut (see Phase 1.8).
- If HF download fails, Phase 3.1 falls back to Drive shortcuts.

**6. Disk space warning**
- The streaming loop checks free space and deletes completed sequences automatically.
- If you see 'Low disk space', wait for the cleanup to finish, then re-run the cell.

### File Locations

| Item | Path |
|------|------|
| Workspace | `/content/sdstrack` |
| Data (ephemeral) | `/content/sdstrack/data/visevent/test/test_subset` |
| Results (persisted to Drive) | `/content/drive/MyDrive/EvTrack/sdstrack_results/VisEvent/cvpr2024_rgbe/` |
| Progress (persisted to Drive) | `/content/drive/MyDrive/EvTrack/sdstrack_progress.json` |
| Pretrained model | `/content/sdstrack/pretrained/...` |
| Checkpoint | `/content/sdstrack/output/checkpoints/...` |